# TP Segmentación Biomédica con U-Net (CVC-ClinicDB)
Notebook compatible con Google Colab.

In [ ]:
# Si hace falta, instalar dependencias en Colab
!pip -q install torch torchvision scikit-learn pillow matplotlib tqdm

In [ ]:
import base64
import io
import json
import random
import zlib
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

## Dataset + Google Drive
Subí `footbal.zip`. El notebook monta Drive, crea `MyDrive/tp3` si no existe, descomprime el dataset ahí y guarda todos los resultados en esa carpeta.

In [ ]:
from google.colab import drive, files
import zipfile
import shutil

drive.mount('/content/drive')

TP3_ROOT = Path('/content/drive/MyDrive/tp3')
DATASET_DIR = TP3_ROOT / 'dataset_footbal'
OUT_DIR = TP3_ROOT / 'outputs'
FIG_DIR = OUT_DIR / 'figures'
PRED_DIR = OUT_DIR / 'predictions'
SHOWCASE_DIR = TP3_ROOT / 'entrega_docente'

for d in [TP3_ROOT, DATASET_DIR, OUT_DIR, FIG_DIR, PRED_DIR, SHOWCASE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
zip_name = next((k for k in uploaded.keys() if k.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError('No se subio ningun .zip')

zip_target = TP3_ROOT / zip_name
shutil.copy(zip_name, zip_target)

with zipfile.ZipFile(zip_target, 'r') as zf:
    zf.extractall(DATASET_DIR)

IMG_DIR = DATASET_DIR / 'images'
COCO_JSON = DATASET_DIR / 'COCO_Football Pixel.json'
if not COCO_JSON.exists():
    raise FileNotFoundError('No se encontro COCO_Football Pixel.json')

coco = json.loads(COCO_JSON.read_text())
TARGET_CLASS_NAME = 'Ball'  # cambiar si queres otra clase

name_to_id = {c['name']: c['id'] for c in coco['categories']}
if TARGET_CLASS_NAME not in name_to_id:
    raise ValueError(f'Clase no encontrada: {TARGET_CLASS_NAME}')
TARGET_CLASS_ID = name_to_id[TARGET_CLASS_NAME]

ann_by_img = {}
for ann in coco['annotations']:
    ann_by_img.setdefault(ann['image_id'], []).append(ann)

imgname_to_id = {im['file_name']: im['id'] for im in coco['images']}

print('TP3_ROOT:', TP3_ROOT)
print('IMG_DIR exists:', IMG_DIR.exists())
print('TARGET_CLASS_NAME:', TARGET_CLASS_NAME)
print('TARGET_CLASS_ID:', TARGET_CLASS_ID)

In [ ]:
def rle_to_mask(seg, height, width):
    counts = seg.get('counts', [])
    if isinstance(counts, str):
        raise ValueError('RLE comprimido (string) no soportado en este notebook')

    flat = np.zeros(height * width, dtype=np.uint8)
    idx = 0
    val = 0
    for run in counts:
        if run > 0:
            flat[idx:idx+run] = val
            idx += run
        val = 1 - val

    return flat.reshape((height, width), order='F')

def polygons_to_mask(polygons, height, width):
    mask_img = Image.new('L', (width, height), 0)
    draw = ImageDraw.Draw(mask_img)
    for poly in polygons:
        if len(poly) < 6:
            continue
        pts = [(poly[i], poly[i+1]) for i in range(0, len(poly), 2)]
        draw.polygon(pts, outline=1, fill=1)
    return np.asarray(mask_img, dtype=np.uint8)

def segmentation_to_mask(seg, height, width):
    if isinstance(seg, dict) and 'counts' in seg:
        return rle_to_mask(seg, height, width)
    if isinstance(seg, list):
        return polygons_to_mask(seg, height, width)
    return np.zeros((height, width), dtype=np.uint8)

def build_target_mask_from_coco(image_file_name, width, height, target_class_id):
    image_id = imgname_to_id.get(image_file_name)
    if image_id is None:
        return np.zeros((height, width), dtype=np.uint8)

    mask = np.zeros((height, width), dtype=np.uint8)
    anns = ann_by_img.get(image_id, [])
    for ann in anns:
        if ann.get('category_id') != target_class_id:
            continue
        seg = ann.get('segmentation', None)
        if seg is None:
            continue
        m = segmentation_to_mask(seg, height, width)
        mask = np.maximum(mask, m.astype(np.uint8))

    return mask


In [ ]:
img_paths = sorted([p for p in IMG_DIR.glob('*.jpg') if '___' not in p.name])
pairs = [(p, p.name) for p in img_paths]

print('Total muestras:', len(pairs))
if len(pairs) == 0:
    raise RuntimeError('No se encontraron imagenes .jpg en images/.')
print('Ejemplo:', pairs[0][0].name)

In [ ]:
def compute_dataset_mean_std(img_paths):
    s1 = np.zeros(3, dtype=np.float64)
    s2 = np.zeros(3, dtype=np.float64)
    n = 0
    for p in img_paths:
        arr = np.asarray(Image.open(p).convert('RGB'), dtype=np.float32) / 255.0
        arr = arr.reshape(-1, 3)
        s1 += arr.sum(axis=0)
        s2 += (arr ** 2).sum(axis=0)
        n += arr.shape[0]
    mean = s1 / n
    std = np.sqrt(np.maximum(s2 / n - mean ** 2, 1e-12))
    return mean.astype(np.float32), std.astype(np.float32)

train_pairs, test_pairs = train_test_split(pairs, test_size=0.2, random_state=42)
train_pairs, val_pairs = train_test_split(train_pairs, test_size=0.2, random_state=42)

mean, std = compute_dataset_mean_std([p for p, _ in train_pairs])
print('Split train/val/test:', len(train_pairs), len(val_pairs), len(test_pairs))
print('mean:', mean, 'std:', std)

In [ ]:
class PolypDataset(Dataset):
    def __init__(self, pairs, image_size=(256, 256), mean=None, std=None, augment=False):
        self.pairs = pairs
        self.image_size = image_size
        self.mean = np.asarray(mean, dtype=np.float32)
        self.std = np.asarray(std, dtype=np.float32)
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, image_file_name = self.pairs[idx]
        image = Image.open(img_path).convert('RGB')
        w, h = image.size
        mask = build_target_mask_from_coco(image_file_name, w, h, TARGET_CLASS_ID).astype(np.uint8)
        mask_img = Image.fromarray(mask * 255)

        image = image.resize(self.image_size, Image.BILINEAR)
        mask_img = mask_img.resize(self.image_size, Image.NEAREST)

        image_np = np.asarray(image, dtype=np.float32) / 255.0
        mask_np = (np.asarray(mask_img, dtype=np.uint8) > 0).astype(np.float32)

        if self.augment:
            if random.random() < 0.5:
                image_np = np.fliplr(image_np).copy()
                mask_np = np.fliplr(mask_np).copy()
            if random.random() < 0.5:
                image_np = np.flipud(image_np).copy()
                mask_np = np.flipud(mask_np).copy()

        image_np = (image_np - self.mean) / self.std
        image_t = torch.from_numpy(image_np.transpose(2, 0, 1)).float()
        mask_t = torch.from_numpy(mask_np).unsqueeze(0).float()
        return image_t, mask_t

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=(64, 128, 256, 512)):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)

        c = in_channels
        for f in features:
            self.downs.append(DoubleConv(c, f))
            c = f

        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        for f in reversed(features):
            self.ups.append(nn.ConvTranspose2d(f * 2, f, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(f * 2, f))

        self.final = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skips = skips[::-1]

        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip = skips[i // 2]
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
            x = torch.cat([skip, x], dim=1)
            x = self.ups[i + 1](x)

        return self.final(x)

In [ ]:
def dice_iou_from_logits(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()
    targets = targets.float()

    inter = (preds * targets).sum(dim=(1, 2, 3))
    pred_area = preds.sum(dim=(1, 2, 3))
    true_area = targets.sum(dim=(1, 2, 3))
    union = pred_area + true_area - inter

    dice = ((2 * inter + eps) / (pred_area + true_area + eps)).mean().item()
    iou = ((inter + eps) / (union + eps)).mean().item()
    acc = (preds == targets).float().mean().item()
    return dice, iou, acc

def evaluate(model, loader, device, criterion):
    model.eval()
    total_loss = 0.0
    dices, ious, accs = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
            d, i, a = dice_iou_from_logits(logits, y)
            dices.append(d)
            ious.append(i)
            accs.append(a)
    n = len(loader.dataset)
    return total_loss / n, float(np.mean(dices)), float(np.mean(ious)), float(np.mean(accs))

In [ ]:
IMAGE_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 12
LR = 1e-3

train_ds = PolypDataset(train_pairs, image_size=(IMAGE_SIZE, IMAGE_SIZE), mean=mean, std=std, augment=True)
val_ds = PolypDataset(val_pairs, image_size=(IMAGE_SIZE, IMAGE_SIZE), mean=mean, std=std, augment=False)
test_ds = PolypDataset(test_pairs, image_size=(IMAGE_SIZE, IMAGE_SIZE), mean=mean, std=std, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = {'train_loss': [], 'val_loss': [], 'val_dice': [], 'val_iou': [], 'val_accuracy': []}
best_val = float('inf')

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running += loss.item() * x.size(0)

    train_loss = running / len(train_loader.dataset)
    val_loss, val_dice, val_iou, val_accuracy = evaluate(model, val_loader, device, criterion)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_dice'].append(val_dice)
    history['val_iou'].append(val_iou)
    history['val_accuracy'].append(val_accuracy)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), OUT_DIR / 'best_unet.pt')

    print(f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_dice={val_dice:.4f} | val_iou={val_iou:.4f} | val_acc={val_accuracy:.4f}')


In [ ]:
model.load_state_dict(torch.load(OUT_DIR / 'best_unet.pt', map_location=device))
test_loss, test_dice, test_iou, test_accuracy = evaluate(model, test_loader, device, criterion)
print('Test loss:', test_loss)
print('Test Dice:', test_dice)
print('Test IoU:', test_iou)
print('Test Accuracy:', test_accuracy)

In [ ]:
epochs = np.arange(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].plot(epochs, history['train_loss'], label='Train')
axes[0,0].plot(epochs, history['val_loss'], label='Val')
axes[0,0].set_title('Loss')
axes[0,0].legend()

axes[0,1].plot(epochs, history['val_dice'], label='Val Dice')
axes[0,1].set_title('Dice')
axes[0,1].legend()

axes[1,0].plot(epochs, history['val_iou'], label='Val IoU')
axes[1,0].set_title('IoU')
axes[1,0].legend()

axes[1,1].plot(epochs, history['val_accuracy'], label='Val Accuracy')
axes[1,1].set_title('Val Accuracy')
axes[1,1].legend()

for ax in axes.flatten():
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'training_curves.png', dpi=150)
plt.show()

In [ ]:
def denormalize(img_t, mean, std):
    x = img_t.cpu().numpy().transpose(1, 2, 0)
    x = (x * std) + mean
    return np.clip(x, 0, 1)

# Configurable export settings
MAX_PRED_SAMPLES = 40
PRED_THRESHOLD = 0.30

model.eval()
shown = 0
with torch.no_grad():
    for x, y in test_loader:
        x_device = x.to(device)
        logits = model(x_device)
        probs = torch.sigmoid(logits).cpu()
        preds = (probs > PRED_THRESHOLD).float()

        for b in range(x.shape[0]):
            if shown >= MAX_PRED_SAMPLES:
                break
            img = denormalize(x[b], mean, std)
            gt = y[b, 0].cpu().numpy()
            pr = preds[b, 0].numpy()
            pr_prob = probs[b, 0].numpy()

            fig, axes = plt.subplots(1, 4, figsize=(16, 4))
            axes[0].imshow(img)
            axes[0].set_title('Imagen test')
            axes[1].imshow(gt, cmap='gray', vmin=0, vmax=1)
            axes[1].set_title('Mascara real')
            axes[2].imshow(pr_prob, cmap='magma', vmin=0, vmax=1)
            axes[2].set_title('Probabilidad')
            axes[3].imshow(pr, cmap='gray', vmin=0, vmax=1)
            axes[3].set_title(f'Prediccion (thr={PRED_THRESHOLD})')
            for ax in axes:
                ax.axis('off')
            plt.tight_layout()
            plt.savefig(PRED_DIR / f'pred_{shown:03d}.png', dpi=150)
            plt.show()
            plt.close(fig)
            shown += 1

        if shown >= MAX_PRED_SAMPLES:
            break

print(f'Se guardaron {shown} visualizaciones en {PRED_DIR}')

In [ ]:
summary = {
    'dataset': 'COCO_Football Pixel (Ball class)',
    'num_total': len(pairs),
    'num_train': len(train_pairs),
    'num_val': len(val_pairs),
    'num_test': len(test_pairs),
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'learning_rate': LR,
    'mean': mean.tolist(),
    'std': std.tolist(),
    'test_loss': test_loss,
    'test_dice': test_dice,
    'test_iou': test_iou
    ,'test_accuracy': test_accuracy
}

(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2))
(OUT_DIR / 'history.json').write_text(json.dumps(history, indent=2))
print(json.dumps(summary, indent=2))
# Copia de salida para mostrar al docente
import shutil
shutil.copy2(FIG_DIR / 'training_curves.png', SHOWCASE_DIR / 'training_curves.png')
for p in sorted(PRED_DIR.glob('pred_*.png')):
    shutil.copy2(p, SHOWCASE_DIR / p.name)
shutil.copy2(OUT_DIR / 'summary.json', SHOWCASE_DIR / 'summary.json')
shutil.copy2(OUT_DIR / 'history.json', SHOWCASE_DIR / 'history.json')
print('Entrega docente guardada en:', SHOWCASE_DIR)